Policy RAG Assistant and Refund Agent

In [1]:
import sys
import pandas as pd
import numpy as np

import sqlite3
import json
import re
import math
import time
import uuid

from pathlib import Path
from datetime import datetime, timezone

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
import mock_llm


PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

POLICY_DIR = DATA_DIR / "policies"

RAW_DIR = DATA_DIR / "raw"

DB_PATH = DATA_DIR / "instamart.db"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

STATE_DIR = PROJECT_ROOT / "state"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATE_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
print("Project root:", PROJECT_ROOT)
print("Policy directory:", POLICY_DIR)
print("Database:", DB_PATH)
print("Output directory:", OUTPUT_DIR)
print("State directory:", STATE_DIR)

assert POLICY_DIR.exists()
assert DB_PATH.exists()

Project root: c:\Users\srira\Documents\personal\ML\instamart_copilot
Policy directory: c:\Users\srira\Documents\personal\ML\instamart_copilot\data\policies
Database: c:\Users\srira\Documents\personal\ML\instamart_copilot\data\instamart.db
Output directory: c:\Users\srira\Documents\personal\ML\instamart_copilot\outputs
State directory: c:\Users\srira\Documents\personal\ML\instamart_copilot\state


In [3]:
policy_files = sorted(
    POLICY_DIR.glob("*.txt")
)

print("Policy documents found:", len(policy_files))

for file in policy_files:
    print(file.name)

assert len(policy_files) == 5

Policy documents found: 5
delivery_policy.txt
escalation_policy.txt
payment_policy.txt
privacy_policy.txt
refund_policy.txt


In [4]:
def chunk_document(text, source, chunk_size=40, overlap=10):

    words = text.split()

    chunks = []

    step = chunk_size - overlap

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk_words = words[start:end]

        chunk_text = " ".join(chunk_words)

        chunks.append({
            "source": source,
            "text": chunk_text,
            "start_word": start,
            "end_word": min(end, len(words)) - 1
        })

        # Stop after the first chunk reaching the final word.
        if end >= len(words):
            break

        start += step

    return chunks

In [5]:
all_chunks = []

for file_path in policy_files:

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        document_text = file.read()

    document_chunks = chunk_document(
        text=document_text,
        source=file_path.name,
        chunk_size=40,
        overlap=10
    )

    all_chunks.extend(document_chunks)


chunks_overlap_10 = pd.DataFrame(all_chunks)

print("Total chunks:", len(chunks_overlap_10))

display(chunks_overlap_10)

Total chunks: 15


,source,text,start_word,end_word
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...,0,39
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ...",30,69
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...,60,99
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...,0,39
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours....",30,69
5,escalation_policy.txt,a short summary. A human agent must call the c...,60,77
6,payment_policy.txt,"Payment Policy. Customers can pay using UPI, d...",0,39
7,payment_policy.txt,"fails but money is deducted, the amount is ret...",30,69
8,payment_policy.txt,identifier. Wallet balance can be used for any...,60,85
9,privacy_policy.txt,Privacy Policy. Customer phone numbers and ema...,0,39


In [6]:
chunk_counts = (
    chunks_overlap_10
    .groupby("source")
    .size()
    .sort_index()
)

print(chunk_counts)

print("Total:", chunk_counts.sum())

source
delivery_policy.txt      3
escalation_policy.txt    3
payment_policy.txt       3
privacy_policy.txt       2
refund_policy.txt        4
dtype: int64
Total: 15


In [7]:
expected_chunk_counts = {
    "refund_policy.txt": 4,
    "delivery_policy.txt": 3,
    "escalation_policy.txt": 3,
    "payment_policy.txt": 3,
    "privacy_policy.txt": 2
}

assert chunk_counts.to_dict() == expected_chunk_counts

assert len(chunks_overlap_10) == 15

In [8]:
vectorizer = TfidfVectorizer()

chunk_texts = chunks_overlap_10["text"].tolist()

tfidf_matrix = vectorizer.fit_transform(
    chunk_texts
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (15, 200)


In [9]:
def retrieve(query, k=3):

    query_vector = vectorizer.transform([
        query
    ])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(
        similarity_scores
    )[::-1][:k]

    results = []

    for index in top_indices:

        results.append({

            "source": chunks_overlap_10.iloc[index]["source"],

            "score": float(
                similarity_scores[index]
            ),

            "text": chunks_overlap_10.iloc[index]["text"]

        })

    return results

In [10]:
test_results = retrieve(
    "refund policy late delivery",
    k=3
)

for rank, result in enumerate(test_results, start=1):

    print(f"\nRank: {rank}")

    print("Source:", result["source"])

    print("Score:", round(result["score"], 4))

    print("Text:", result["text"])


Rank: 1
Source: refund_policy.txt
Score: 0.2719
Text: order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are

Rank: 2
Source: refund_policy.txt
Score: 0.2514
Text: Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer

Rank: 3
Source: refund_policy.txt
Score: 0.2454
Text: late-delivery refund is paid. Refunds up to INR 1000 are approved automatically by the support system. A refund above INR 1000 must be approved by a human support lead before money is returned. Approved refunds are credited to the original


In [11]:
assert all(
    result["source"] == "refund_policy.txt"
    for result in test_results)

In [12]:
POLICY_SYSTEM_PROMPT = """
You are a POLICY ASSISTANT for Instamart customer support.

Answer the question using only the supplied policy context.

Rules:
1. Use only information present in the supplied context.
2. Do not use external knowledge or invent policy details.
3. If the context does not contain the answer, respond exactly:
   I could not find that information in the policy documents.
4. Keep the answer clear and concise.
5. Do not make assumptions beyond the policy text.
"""

In [13]:
def answer(question):

    retrieved_chunks = retrieve(
        question,
        k=3
    )

    context_parts = []

    for chunk in retrieved_chunks:

        context_parts.append(
            f"[{chunk['source']}]\n{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    user_message = (
        "CONTEXT:\n"
        + context
        + "\nQUESTION:\n"
        + question
    )

    messages = [
        {
            "role": "system",
            "content": POLICY_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    response = mock_llm.chat(messages)

    model_answer = response.choices[0].message.content

    sources_used = list(dict.fromkeys(
        chunk["source"]
        for chunk in retrieved_chunks
    ))

    return {
        "answer": model_answer,
        "sources": sources_used
    }

In [14]:
test_question = (
    "What is the refund policy for late delivery?"
)

result = answer(test_question)

print("Answer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Answer:
Refunds up to INR 1000 are

 
Refund Policy.

Sources:
['refund_policy.txt']


In [16]:
policy_eval_df = pd.read_csv(
    RAW_DIR / "policy_eval.csv"
)

print(policy_eval_df.shape)

print(policy_eval_df.columns.tolist())

display(policy_eval_df)

(14, 3)
['question', 'expected_doc', 'expected_fact']


,question,expected_doc,expected_fact
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days


In [17]:
required_columns = {
    "question",
    "expected_doc",
    "expected_fact"
}

assert required_columns.issubset(
    policy_eval_df.columns
)

In [18]:
answerable_df = policy_eval_df[
    policy_eval_df["expected_doc"].notna()
].copy()

unanswerable_df = policy_eval_df[
    policy_eval_df["expected_doc"].isna()
].copy()

print("Answerable:", len(answerable_df))

print("Unanswerable:", len(unanswerable_df))

Answerable: 14
Unanswerable: 0


In [19]:
def evaluate_policy_retrieval(eval_data, k=3):

    results = []

    for _, row in eval_data.iterrows():

        question = str(row["question"])

        expected_doc = row["expected_doc"]

        expected_fact = row["expected_fact"]

        retrieved = retrieve(
            question,
            k=k
        )

        retrieved_sources = [
            chunk["source"]
            for chunk in retrieved
        ]

        model_result = answer(question)

        model_text = model_result["answer"]

        if pd.notna(expected_doc):

            doc_found = expected_doc in retrieved_sources

            fact_found = (
                str(expected_fact) in model_text
            )

            fallback_found = None

        else:

            doc_found = None

            fact_found = None

            fallback_found = model_text.startswith(
                "I could not find"
            )

        results.append({

            "question": question,

            "expected_doc": expected_doc,

            "expected_fact": expected_fact,

            "retrieved_sources": retrieved_sources,

            "answer": model_text,

            "doc_found": doc_found,

            "fact_found": fact_found,

            "fallback_found": fallback_found

        })

    return pd.DataFrame(results)

In [20]:
eval_overlap_10 = evaluate_policy_retrieval(
    policy_eval_df
)

display(eval_overlap_10)

,question,expected_doc,expected_fact,retrieved_sources,answer,doc_found,fact_found,fallback_found
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes,"[refund_policy.txt, refund_policy.txt, privacy...","order is delivered in more than 45 minutes, th...",True,True,None
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead,"[refund_policy.txt, refund_policy.txt, refund_...",A refund above INR 1000 must be approved by a ...,True,True,None
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days,"[refund_policy.txt, privacy_policy.txt, refund...",A refund request must be raised within 7 days ...,True,True,None
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25,"[delivery_policy.txt, delivery_policy.txt, pay...",A delivery fee of INR 25 is charged on orders ...,True,True,None
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more,"[delivery_policy.txt, delivery_policy.txt, pay...","of INR 25 is charged on orders below INR 199, ...",True,True,None
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password,"[delivery_policy.txt, delivery_policy.txt, pay...",Electronics items are delivered only after the...,True,True,None
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500,"[payment_policy.txt, payment_policy.txt, payme...",Cash on delivery is available only for orders ...,True,True,None
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours,"[payment_policy.txt, payment_policy.txt, refun...","If a payment fails but money is deducted, the ...",True,True,None
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash,"[payment_policy.txt, payment_policy.txt, payme...",Wallet balance can be used for any order and i...,True,True,None
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days,"[privacy_policy.txt, privacy_policy.txt, deliv...",Delivery addresses are deleted from active sys...,True,True,None


In [21]:
answerable_results = eval_overlap_10[
    eval_overlap_10["expected_doc"].notna()
]

unanswerable_results = eval_overlap_10[
    eval_overlap_10["expected_doc"].isna()
]

overlap_10_summary = {

    "Answerable questions": len(answerable_results),

    "Expected document in top 3": int(
        answerable_results["doc_found"].sum()
    ),

    "Expected fact in answer": int(
        answerable_results["fact_found"].sum()
    ),

    "Unanswerable questions": len(unanswerable_results),

    "Correct fallback answers": int(
        unanswerable_results["fallback_found"].sum()
    )

}

print(overlap_10_summary)

{'Answerable questions': 14, 'Expected document in top 3': 12, 'Expected fact in answer': 12, 'Unanswerable questions': 0, 'Correct fallback answers': 0}


In [22]:
all_chunks_zero = []

for file_path in policy_files:

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:

        document_text = file.read()

    document_chunks = chunk_document(
        text=document_text,
        source=file_path.name,
        chunk_size=40,
        overlap=0
    )

    all_chunks_zero.extend(document_chunks)


chunks_overlap_0 = pd.DataFrame(
    all_chunks_zero
)

print("Zero-overlap chunk count:", len(chunks_overlap_0))

Zero-overlap chunk count: 13


In [23]:
vectorizer_zero = TfidfVectorizer()

tfidf_matrix_zero = vectorizer_zero.fit_transform(
    chunks_overlap_0["text"].tolist()
)

In [24]:
def retrieve_zero(query, k=3):

    query_vector = vectorizer_zero.transform([
        query
    ])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix_zero
    ).flatten()

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for index in top_indices:

        results.append({

            "source": chunks_overlap_0.iloc[index]["source"],

            "score": float(scores[index]),

            "text": chunks_overlap_0.iloc[index]["text"]

        })

    return results

In [25]:
def answer_zero(question):

    retrieved_chunks = retrieve_zero(
        question,
        k=3
    )

    context_parts = []

    for chunk in retrieved_chunks:

        context_parts.append(
            f"[{chunk['source']}]\n{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    user_message = (
        "CONTEXT:\n"
        + context
        + "\nQUESTION:\n"
        + question
    )

    messages = [
        {
            "role": "system",
            "content": POLICY_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    response = mock_llm.chat(messages)

    return {
        "answer": response.choices[0].message.content,
        "sources": list(dict.fromkeys(
            chunk["source"]
            for chunk in retrieved_chunks
        ))
    }

In [26]:
def evaluate_policy_zero(eval_data):

    results = []

    for _, row in eval_data.iterrows():

        question = str(row["question"])

        expected_doc = row["expected_doc"]

        expected_fact = row["expected_fact"]

        retrieved = retrieve_zero(
            question,
            k=3
        )

        sources = [
            chunk["source"]
            for chunk in retrieved
        ]

        model_result = answer_zero(question)

        model_text = model_result["answer"]

        if pd.notna(expected_doc):

            doc_found = expected_doc in sources

            fact_found = str(expected_fact) in model_text

            fallback_found = None

        else:

            doc_found = None

            fact_found = None

            fallback_found = model_text.startswith(
                "I could not find"
            )

        results.append({

            "question": question,

            "expected_doc": expected_doc,

            "expected_fact": expected_fact,

            "retrieved_sources": sources,

            "answer": model_text,

            "doc_found": doc_found,

            "fact_found": fact_found,

            "fallback_found": fallback_found

        })

    return pd.DataFrame(results)

In [27]:
eval_overlap_0 = evaluate_policy_zero(
    policy_eval_df
)

display(eval_overlap_0)

,question,expected_doc,expected_fact,retrieved_sources,answer,doc_found,fact_found,fallback_found
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes,"[refund_policy.txt, refund_policy.txt, privacy...",receives a full refund of the order amount.,True,False,None
1,Who must approve a refund above INR 1000?,refund_policy.txt,human support lead,"[refund_policy.txt, refund_policy.txt, refund_...",A refund above INR\n\n \n1000 must be approved...,True,True,None
2,How many days do I have to raise a refund requ...,refund_policy.txt,within 7 days,"[refund_policy.txt, refund_policy.txt, refund_...",A refund request must be raised within 7 days ...,True,True,None
3,What is the delivery fee on small orders?,delivery_policy.txt,INR 25,"[delivery_policy.txt, payment_policy.txt, deli...",A delivery fee of INR 25 is charged on orders ...,True,True,None
4,On orders of what value is delivery free?,delivery_policy.txt,INR 199 or more,"[delivery_policy.txt, delivery_policy.txt, pay...",A delivery fee of INR 25 is charged on orders ...,True,True,None
5,Why does the electronics delivery need a one t...,delivery_policy.txt,one time password,"[delivery_policy.txt, delivery_policy.txt, del...",Electronics items are delivered only after the...,True,True,None
6,Is cash on delivery available for a large order?,payment_policy.txt,INR 2500,"[payment_policy.txt, payment_policy.txt, payme...",Cash on delivery is available only for orders ...,True,True,None
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours,"[payment_policy.txt, refund_policy.txt, privac...","If a payment fails but money is deducted, the ...",True,False,None
8,Can wallet balance be refunded as cash?,payment_policy.txt,never refunded as cash,"[payment_policy.txt, payment_policy.txt, payme...",Wallet balance can be used for any order and i...,True,True,None
9,When are delivery addresses deleted from activ...,privacy_policy.txt,180 days,"[privacy_policy.txt, delivery_policy.txt, esca...",Delivery addresses are deleted from active sys...,True,True,None


In [28]:
answerable_zero = eval_overlap_0[
    eval_overlap_0["expected_doc"].notna()
]

unanswerable_zero = eval_overlap_0[
    eval_overlap_0["expected_doc"].isna()
]

overlap_0_summary = {

    "Expected document in top 3": int(
        answerable_zero["doc_found"].sum()
    ),

    "Expected fact in answer": int(
        answerable_zero["fact_found"].sum()
    ),

    "Correct fallback answers": int(
        unanswerable_zero["fallback_found"].sum()
    )

}

print(overlap_0_summary)

{'Expected document in top 3': 12, 'Expected fact in answer': 10, 'Correct fallback answers': 0}


In [29]:
failed_questions = eval_overlap_0[
    (
        eval_overlap_0["expected_doc"].notna()
    )
    &
    (
        (eval_overlap_0["doc_found"] == False)
        |
        (eval_overlap_0["fact_found"] == False)
    )
]

display(
    failed_questions[
        [
            "question",
            "expected_doc",
            "expected_fact",
            "retrieved_sources",
            "answer"
        ]
    ]
)

,question,expected_doc,expected_fact,retrieved_sources,answer
0,What delivery time makes a completed order eli...,refund_policy.txt,more than 45 minutes,"[refund_policy.txt, refund_policy.txt, privacy...",receives a full refund of the order amount.
7,How soon is money returned when a payment fail...,payment_policy.txt,48 hours,"[payment_policy.txt, refund_policy.txt, privac...","If a payment fails but money is deducted, the ..."
12,What is the favourite colour of the company ma...,NONE,NaN,"[delivery_policy.txt, delivery_policy.txt, ref...",I could not find that information in the polic...
13,Who won the cricket world cup final?,NONE,NaN,"[payment_policy.txt, privacy_policy.txt, deliv...",I could not find that information in the polic...


In [30]:
failed_question = failed_questions.iloc[0]["question"]

expected_document = failed_questions.iloc[0]["expected_doc"]

print("Question:", failed_question)

print("Expected document:", expected_document)

display(
    chunks_overlap_0[
        chunks_overlap_0["source"] == expected_document
    ][
        ["start_word", "end_word", "text"]
    ]
)

Question: What delivery time makes a completed order eligible for a full refund?
Expected document: refund_policy.txt


,start_word,end_word,text
10,0,39,Refund Policy. A refund can be requested only ...
11,40,79,receives a full refund of the order amount. If...
12,80,118,1000 must be approved by a human support lead ...


In [31]:
comparison_df = pd.DataFrame([

    {
        "Chunk overlap": 10,
        "Expected document found": (
            f"{int(answerable_results['doc_found'].sum())}/12"
        ),
        "Expected fact found": (
            f"{int(answerable_results['fact_found'].sum())}/12"
        ),
        "Correct fallbacks": (
            f"{int(unanswerable_results['fallback_found'].sum())}/2"
        )
    },

    {
        "Chunk overlap": 0,
        "Expected document found": (
            f"{int(answerable_zero['doc_found'].sum())}/12"
        ),
        "Expected fact found": (
            f"{int(answerable_zero['fact_found'].sum())}/12"
        ),
        "Correct fallbacks": (
            f"{int(unanswerable_zero['fallback_found'].sum())}/2"
        )
    }

])

display(comparison_df)

,Chunk overlap,Expected document found,Expected fact found,Correct fallbacks
0,10,12/12,12/12,0/2
1,0,12/12,10/12,0/2


In [32]:
connection = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    connection
)

display(tables_df)

for table_name in tables_df["name"]:

    print("\nTable:", table_name)

    schema_df = pd.read_sql_query(
        f"PRAGMA table_info('{table_name}')",
        connection
    )

    display(schema_df)

connection.close()

,name
0,customers
1,products
2,orders



Table: customers


,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,TEXT,0,None,0
1,1,city,TEXT,0,None,0
2,2,signup_date,TIMESTAMP,0,None,0
3,3,age,REAL,0,None,0
4,4,gender,TEXT,0,None,0



Table: products


,cid,name,type,notnull,dflt_value,pk
0,0,product_id,TEXT,0,None,0
1,1,product_name,TEXT,0,None,0
2,2,category,TEXT,0,None,0
3,3,price_inr,INTEGER,0,None,0



Table: orders


,cid,name,type,notnull,dflt_value,pk
0,0,order_id,TEXT,0,None,0
1,1,customer_id,TEXT,0,None,0
2,2,product_id,TEXT,0,None,0
3,3,quantity,INTEGER,0,None,0
4,4,order_ts,TEXT,0,None,0
5,5,status,TEXT,0,None,0
6,6,discount_pct,INTEGER,0,None,0
7,7,order_amount,REAL,0,None,0
8,8,delivery_minutes,REAL,0,None,0
9,9,category,TEXT,0,None,0


In [33]:
def get_order(order_id):

    connection = sqlite3.connect(DB_PATH)

    try:

        query = """
            SELECT
                order_id,
                status,
                order_amount,
                delivery_minutes
            FROM orders
            WHERE order_id = ?
            LIMIT 1
        """

        result = pd.read_sql_query(
            query,
            connection,
            params=(order_id,)
        )

        if result.empty:

            return {
                "error": "order not found"
            }

        row = result.iloc[0]

        return {

            "order_id": str(row["order_id"]),

            "status": str(row["status"]).strip().lower(),

            "order_amount": float(row["order_amount"]),

            "delivery_minutes": (
                None
                if pd.isna(row["delivery_minutes"])
                else float(row["delivery_minutes"])
            )

        }

    finally:

        connection.close()

In [34]:
def search_policy(query):

    return {
        "results": retrieve(
            query,
            k=3
        )
    }

In [35]:
def calculate_refund(
    order_amount,
    delivery_minutes
):

    order_amount = float(order_amount)

    delivery_minutes = float(delivery_minutes)

    if delivery_minutes > 45:

        refund_amount = order_amount

    else:

        refund_amount = 0.0

    return {
        "refund_inr": round(
            refund_amount,
            2
        )
    }

In [36]:
print(
    calculate_refund(
        order_amount=128.25,
        delivery_minutes=60
    )
)

{'refund_inr': 128.25}


In [37]:
REFUND_LEDGER_PATH = OUTPUT_DIR / "refund_ledger.jsonl"


def append_refund_ledger(
    order_id,
    refund_inr,
    tool_name
):

    record = {

        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),

        "order_id": str(order_id),

        "refund_inr": round(
            float(refund_inr),
            2
        ),

        "tool_name": tool_name

    }

    with open(
        REFUND_LEDGER_PATH,
        "a",
        encoding="utf-8"
    ) as file:

        file.write(
            json.dumps(record) + "\n"
        )

    return {
        "refund_inr": record["refund_inr"]
    }

In [38]:
def issue_refund(
    order_id,
    refund_inr
):

    return append_refund_ledger(
        order_id=order_id,
        refund_inr=refund_inr,
        tool_name="issue_refund"
    )


def escalate_to_human(
    order_id,
    refund_inr
):

    return append_refund_ledger(
        order_id=order_id,
        refund_inr=refund_inr,
        tool_name="escalate_to_human"
    )

In [39]:
TOOLS_SPEC = [

    {
        "name": "get_order",

        "description": (
            "Look up an order by order ID and return its "
            "status, order amount and delivery time."
        ),

        "parameters": {
            "type": "object",

            "properties": {
                "order_id": {
                    "type": "string",
                    "description": "The order ID."
                }
            },

            "required": ["order_id"]
        }
    },

    {
        "name": "search_policy",

        "description": (
            "Search the policy documents and return "
            "the three most relevant chunks."
        ),

        "parameters": {
            "type": "object",

            "properties": {
                "query": {
                    "type": "string",
                    "description": "The policy search query."
                }
            },

            "required": ["query"]
        }
    },

    {
        "name": "calculate_refund",

        "description": (
            "Calculate the eligible refund from the "
            "order amount and delivery time."
        ),

        "parameters": {
            "type": "object",

            "properties": {
                "order_amount": {
                    "type": "number"
                },

                "delivery_minutes": {
                    "type": "number"
                }
            },

            "required": [
                "order_amount",
                "delivery_minutes"
            ]
        }
    },

    {
        "name": "issue_refund",

        "description": (
            "Record an automatically issued refund "
            "in the refund ledger."
        ),

        "parameters": {
            "type": "object",

            "properties": {
                "order_id": {
                    "type": "string"
                },

                "refund_inr": {
                    "type": "number"
                }
            },

            "required": [
                "order_id",
                "refund_inr"
            ]
        }
    },

    {
        "name": "escalate_to_human",

        "description": (
            "Record a refund requiring human approval "
            "in the refund ledger."
        ),

        "parameters": {
            "type": "object",

            "properties": {
                "order_id": {
                    "type": "string"
                },

                "refund_inr": {
                    "type": "number"
                }
            },

            "required": [
                "order_id",
                "refund_inr"
            ]
        }
    }

]

In [40]:
TOOL_MAP = {

    "get_order": get_order,

    "search_policy": search_policy,

    "calculate_refund": calculate_refund,

    "issue_refund": issue_refund,

    "escalate_to_human": escalate_to_human

}

In [41]:
print(TOOL_MAP.keys())

dict_keys(['get_order', 'search_policy', 'calculate_refund', 'issue_refund', 'escalate_to_human'])


In [42]:
print(
    TOOL_MAP["calculate_refund"](
        order_amount=128.25,
        delivery_minutes=60
    )
)

{'refund_inr': 128.25}


In [43]:
AGENT_SYSTEM_PROMPT = """
You are an AGENT responsible for Instamart refund requests.

Use the available tools to handle refund requests.

Rules:
1. Look up the order using get_order.
2. If the order is not found, return NOT_FOUND.
3. Only completed orders are eligible.
4. Search the relevant refund policy using search_policy.
5. Use calculate_refund to calculate the refund.
6. If the refund is zero, return DENIED.
7. If the refund is above INR 1000, use escalate_to_human.
8. Otherwise, use issue_refund.
9. Never claim a refund was issued unless issue_refund succeeded.
10. Never claim an escalation succeeded unless escalate_to_human succeeded.

All simulated requests are within the policy's 7-day window.

Your final response must begin with one of:
REFUND:
ESCALATE:
DENIED:
NOT_FOUND:
"""

In [45]:
def extract_tool_calls(message):

    raw_tool_calls = message.tool_calls or []

    extracted = []

    for tool_call in raw_tool_calls:

        extracted.append({

            "id": tool_call.id,

            "name": tool_call.name,

            "arguments": tool_call.arguments

        })

    return extracted

In [46]:
def execute_tool(tool_call):

    tool_name = tool_call["name"]

    arguments = tool_call["arguments"]

    if tool_name not in TOOL_MAP:

        raise ValueError(
            f"Unknown tool requested: {tool_name}"
        )

    tool_function = TOOL_MAP[tool_name]

    observation = tool_function(
        **arguments
    )

    return observation

In [47]:
def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    messages = [
        {
            "role": "system",
            "content": AGENT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

    steps = []

    llm_calls = 0

    tokens_used = 0

    start_time = time.monotonic()

    for iteration in range(max_iterations):

        response = mock_llm.chat(
            messages,
            tools=TOOLS_SPEC
        )

        llm_calls += 1

        tokens_used += response.usage.total_tokens

        message = response.choices[0].message

        tool_calls = extract_tool_calls(message)

        # If there are no tool calls, the model has finished.
        if not tool_calls:

            final_answer = message.content or ""

            status = final_answer.split(":")[0].strip()

            return {
                "status": status,
                "final_answer": final_answer,
                "steps": steps,
                "llm_calls": llm_calls,
                "tokens_used": tokens_used
            }

        # Append the assistant's tool-call message.
        assistant_tool_calls = []

        for tool_call in tool_calls:

            assistant_tool_calls.append({

                "id": tool_call["id"],

                "name": tool_call["name"],

                "arguments": tool_call["arguments"]

            })

        messages.append({

            "role": "assistant",

            "content": None,

            "tool_calls": assistant_tool_calls

        })

        # Execute the requested tools.
        for tool_call in tool_calls:

            observation = execute_tool(
                tool_call
            )

            steps.append({

                "tool_name": tool_call["name"],

                "arguments": tool_call["arguments"],

                "observation": observation

            })

            messages.append({

                "role": "tool",

                "tool_call_id": tool_call["id"],

                "name": tool_call["name"],

                "content": json.dumps(observation)

            })

    return {
        "status": "FALLBACK_MAX_ITERATIONS",

        "final_answer": (
            "FALLBACK_MAX_ITERATIONS: "
            "The agent reached its maximum number of model calls. "
            "Please send this request to a human support agent."
        ),

        "steps": steps,

        "llm_calls": llm_calls,

        "tokens_used": tokens_used
    }

In [48]:
scenarios_df = pd.read_csv(
    RAW_DIR / "agent_scenarios.csv"
)

print("Shape:", scenarios_df.shape)

print("Columns:", scenarios_df.columns.tolist())

display(scenarios_df)

Shape: (7, 5)
Columns: ['scenario_id', 'customer_id', 'message', 'expected_outcome', 'expected_refund_inr']


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


In [51]:
print(scenarios_df.columns.tolist())



['scenario_id', 'customer_id', 'message', 'expected_outcome', 'expected_refund_inr']


In [53]:
MESSAGE_COLUMN = "message"

In [54]:
assert "scenario_id" in scenarios_df.columns

assert MESSAGE_COLUMN in scenarios_df.columns

assert len(scenarios_df) == 7

In [55]:
s1_row = scenarios_df[
    scenarios_df["scenario_id"].astype(str) == "S1"
].iloc[0]

s1_message = str(
    s1_row[MESSAGE_COLUMN]
)

mock_llm.reset()

s1_test = run_agent(
    user_message=s1_message,
    scenario_id="S1"
)

print("Status:", s1_test["status"])

print("Final answer:", s1_test["final_answer"])

print("Model calls:", s1_test["llm_calls"])

print("Tokens:", s1_test["tokens_used"])

print("\nTool sequence:")

for step in s1_test["steps"]:
    print(step["tool_name"])

Status: REFUND
Final answer: REFUND: INR 128.25 issued.
Model calls: 5
Tokens: 1597

Tool sequence:
get_order
search_policy
calculate_refund
issue_refund


In [56]:
def checkpoint_path(run_id):

    return STATE_DIR / f"{run_id}.json"


def save_checkpoint(state):

    path = checkpoint_path(
        state["run_id"]
    )

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            state,
            file,
            indent=2,
            default=str
        )


def load_checkpoint(run_id):

    path = checkpoint_path(
        run_id
    )

    if not path.exists():

        raise FileNotFoundError(
            f"Checkpoint not found: {path}"
        )

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as file:

        return json.load(file)

In [57]:
def make_fallback(status, message, state):

    return {

        "status": status,

        "final_answer": f"{status}: {message}",

        "steps": state["steps"],

        "llm_calls": state["llm_calls"],

        "tokens_used": state["tokens_used"],

        "run_id": state["run_id"]

    }

In [58]:
def continue_agent(
    state,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    start_time = state["start_monotonic"]

    while True:

        # ------------------------------------------------
        # 1. Complete any pending tool calls first.
        # ------------------------------------------------

        if state["pending_tool_calls"]:

            if state["pending_index"] < len(
                state["pending_tool_calls"]
            ):

                tool_call = state["pending_tool_calls"][
                    state["pending_index"]
                ]

                observation = execute_tool(
                    tool_call
                )

                # Record the tool execution.
                step = {

                    "tool_name": tool_call["name"],

                    "arguments": tool_call["arguments"],

                    "observation": observation

                }

                state["steps"].append(step)

                state["tools_executed"].append(
                    tool_call["name"]
                )

                state["tool_execution_count"] += 1

                # Append the tool observation to messages.
                state["messages"].append({

                    "role": "tool",

                    "tool_call_id": tool_call["id"],

                    "name": tool_call["name"],

                    "content": json.dumps(observation)

                })

                # Move to the next pending tool.
                state["pending_index"] += 1

                # Save immediately after the observation.
                save_checkpoint(state)

                # Simulated crash for Task 8.
                if (
                    state["crash_after_tool_executions"]
                    is not None
                    and state["tool_execution_count"]
                    == state["crash_after_tool_executions"]
                ):

                    # Disable the crash before resuming.
                    state["crash_after_tool_executions"] = None

                    save_checkpoint(state)

                    raise RuntimeError(
                        "SIMULATED_CRASH_AFTER_CHECKPOINT"
                    )

                continue

            else:

                # All pending tools have been executed.
                state["pending_tool_calls"] = []

                state["pending_index"] = 0

                save_checkpoint(state)

                continue

        # ------------------------------------------------
        # 2. Check elapsed time before the next model call.
        # ------------------------------------------------

        elapsed = time.monotonic() - start_time

        if elapsed >= time_limit_seconds:

            return make_fallback(

                "FALLBACK_TIMEOUT",

                "The agent exceeded its execution time. "
                "Please send this request to a human support agent.",

                state

            )

        # ------------------------------------------------
        # 3. Check maximum model calls.
        # ------------------------------------------------

        if state["llm_calls"] >= max_iterations:

            return make_fallback(

                "FALLBACK_MAX_ITERATIONS",

                "The agent reached its maximum number of model calls. "
                "Please send this request to a human support agent.",

                state

            )

        # ------------------------------------------------
        # 4. Call the mock model.
        # ------------------------------------------------

        response = mock_llm.chat(

            state["messages"],

            tools=TOOLS_SPEC

        )

        state["llm_calls"] += 1

        state["tokens_used"] += int(
            response.usage.total_tokens
        )

        # ------------------------------------------------
        # 5. Check token limit before executing tools.
        # ------------------------------------------------

        if state["tokens_used"] > token_limit:

            save_checkpoint(state)

            return make_fallback(

                "FALLBACK_TOKEN_LIMIT",

                "The agent exceeded its token budget. "
                "Please send this request to a human support agent.",

                state

            )

        # ------------------------------------------------
        # 6. Extract the model message and tool calls.
        # ------------------------------------------------

        message = response.choices[0].message

        tool_calls = extract_tool_calls(message)

        # ------------------------------------------------
        # 7. No tool calls means final answer.
        # ------------------------------------------------

        if not tool_calls:

            final_answer = message.content or ""

            status = final_answer.split(":")[0].strip()

            return {

                "status": status,

                "final_answer": final_answer,

                "steps": state["steps"],

                "llm_calls": state["llm_calls"],

                "tokens_used": state["tokens_used"],

                "run_id": state["run_id"]

            }

        # ------------------------------------------------
        # 8. Append the assistant tool-call message.
        # ------------------------------------------------

        assistant_tool_calls = []

        for tool_call in tool_calls:

            assistant_tool_calls.append({

                "id": tool_call["id"],

                "name": tool_call["name"],

                "arguments": tool_call["arguments"]

            })

        state["messages"].append({

            "role": "assistant",

            "content": None,

            "tool_calls": assistant_tool_calls

        })

        # Store the requested tools before executing them.
        state["pending_tool_calls"] = tool_calls

        state["pending_index"] = 0

        save_checkpoint(state)

In [59]:
def run_agent(
    user_message,
    scenario_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10,
    run_id=None,
    crash_after_tool_executions=None
):

    if run_id is None:

        run_id = str(uuid.uuid4())

    state = {

        "run_id": run_id,

        "scenario_id": str(scenario_id),

        "messages": [

            {
                "role": "system",
                "content": AGENT_SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": user_message
            }

        ],

        "tokens_used": 0,

        "llm_calls": 0,

        "steps": [],

        "tools_executed": [],

        "tool_execution_count": 0,

        "pending_tool_calls": [],

        "pending_index": 0,

        "start_monotonic": time.monotonic(),

        "crash_after_tool_executions": (
            crash_after_tool_executions
        )

    }

    # Save the initial state.
    save_checkpoint(state)

    return continue_agent(

        state,

        max_iterations=max_iterations,

        token_limit=token_limit,

        time_limit_seconds=time_limit_seconds

    )

In [60]:
mock_llm.reset()

AGENT_RUNS_PATH = OUTPUT_DIR / "agent_runs.jsonl"

REFUND_LEDGER_PATH = OUTPUT_DIR / "refund_ledger.jsonl"

if AGENT_RUNS_PATH.exists():
    AGENT_RUNS_PATH.unlink()

if REFUND_LEDGER_PATH.exists():
    REFUND_LEDGER_PATH.unlink()

def append_jsonl(path, record):

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as file:

        file.write(
            json.dumps(record, default=str) + "\n"
        )


In [61]:
scenario_results = []

for _, row in scenarios_df.iterrows():

    scenario_id = str(
        row["scenario_id"]
    )

    user_message = str(
        row[MESSAGE_COLUMN]
    )

    result = run_agent(

        user_message=user_message,

        scenario_id=scenario_id,

        max_iterations=6,

        token_limit=6000,

        time_limit_seconds=10

    )

    scenario_results.append(result)

    log_record = {

        "scenario_id": scenario_id,

        "status": result["status"],

        "final_answer": result["final_answer"],

        "llm_calls": result["llm_calls"],

        "tokens_used": result["tokens_used"],

        "steps": result["steps"]

    }

    append_jsonl(
        AGENT_RUNS_PATH,
        log_record
    )

    print(
        scenario_id,
        result["status"],
        result["final_answer"]
    )

S1 REFUND REFUND: INR 128.25 issued.
S2 REFUND REFUND: INR 135.0 issued.
S3 ESCALATE ESCALATE: refund above INR 1000 sent to a human lead.
S4 DENIED DENIED: delivery was within 45 minutes.
S5 DENIED DENIED: only completed orders can be refunded.
S6 NOT_FOUND NOT_FOUND: no such order.
S7 FALLBACK_MAX_ITERATIONS FALLBACK_MAX_ITERATIONS: The agent reached its maximum number of model calls. Please send this request to a human support agent.


In [62]:
expected_statuses = {

    "S1": "REFUND",

    "S2": "REFUND",

    "S3": "ESCALATE",

    "S4": "DENIED",

    "S5": "DENIED",

    "S6": "NOT_FOUND",

    "S7": "FALLBACK_MAX_ITERATIONS"

}

actual_statuses = {

    str(row["scenario_id"]): result["status"]

    for (_, row), result in zip(
        scenarios_df.iterrows(),
        scenario_results
    )

}

for scenario_id, expected_status in expected_statuses.items():

    actual_status = actual_statuses.get(scenario_id)

    print(
        scenario_id,
        "Expected:", expected_status,
        "Actual:", actual_status
    )

    assert actual_status == expected_status

S1 Expected: REFUND Actual: REFUND
S2 Expected: REFUND Actual: REFUND
S3 Expected: ESCALATE Actual: ESCALATE
S4 Expected: DENIED Actual: DENIED
S5 Expected: DENIED Actual: DENIED
S6 Expected: NOT_FOUND Actual: NOT_FOUND
S7 Expected: FALLBACK_MAX_ITERATIONS Actual: FALLBACK_MAX_ITERATIONS


In [63]:
def extract_refund_amount(result):

    for step in result["steps"]:

        if step["tool_name"] in [
            "issue_refund",
            "escalate_to_human"
        ]:

            return float(
                step["observation"]["refund_inr"]
            )

    if result["status"] == "DENIED":
        return 0.0

    return None

In [64]:
expected_refunds = {

    "S1": 128.25,

    "S2": 135.00,

    "S3": 1999.00,

    "S4": 0.00,

    "S5": 0.00

}

for (_, row), result in zip(
    scenarios_df.iterrows(),
    scenario_results
):

    scenario_id = str(row["scenario_id"])

    if scenario_id in expected_refunds:

        actual_refund = extract_refund_amount(result)

        expected_refund = expected_refunds[scenario_id]

        print(
            scenario_id,
            actual_refund,
            expected_refund
        )

        assert abs(
            actual_refund - expected_refund
        ) <= 0.01

S1 128.25 128.25
S2 135.0 135.0
S3 1999.0 1999.0
S4 0.0 0.0
S5 0.0 0.0


In [65]:
s1_row = scenarios_df[
    scenarios_df["scenario_id"].astype(str) == "S1"
].iloc[0]

s1_message = str(
    s1_row[MESSAGE_COLUMN]
)

token_demo = run_agent(

    user_message=s1_message,

    scenario_id="S1_TOKEN_DEMO",

    max_iterations=6,

    token_limit=200,

    time_limit_seconds=10

)

print("Status:", token_demo["status"])

print("Final answer:", token_demo["final_answer"])

print("Model calls:", token_demo["llm_calls"])

print("Tokens used:", token_demo["tokens_used"])

print("Tools completed:", [
    step["tool_name"]
    for step in token_demo["steps"]
])

Status: FALLBACK_TOKEN_LIMIT
Final answer: FALLBACK_TOKEN_LIMIT: The agent exceeded its token budget. Please send this request to a human support agent.
Model calls: 2
Tokens used: 382
Tools completed: ['get_order']


In [66]:
timeout_demo = run_agent(

    user_message=s1_message,

    scenario_id="S1_TIMEOUT_DEMO",

    max_iterations=6,

    token_limit=6000,

    time_limit_seconds=0

)

print("Status:", timeout_demo["status"])

print("Final answer:", timeout_demo["final_answer"])

print("Model calls:", timeout_demo["llm_calls"])

print("Tokens used:", timeout_demo["tokens_used"])

Status: FALLBACK_TIMEOUT
Final answer: FALLBACK_TIMEOUT: The agent exceeded its execution time. Please send this request to a human support agent.
Model calls: 0
Tokens used: 0


In [67]:
def resume_agent(
    run_id,
    max_iterations=6,
    token_limit=6000,
    time_limit_seconds=10
):

    state = load_checkpoint(run_id)

    print("Resuming run:", state["run_id"])

    print(
        "Previously executed tools:",
        state["tools_executed"]
    )

    # Start a fresh elapsed-time measurement for the resumed process.
    state["start_monotonic"] = time.monotonic()

    return continue_agent(

        state,

        max_iterations=max_iterations,

        token_limit=token_limit,

        time_limit_seconds=time_limit_seconds

    )

In [68]:
crash_demo_run_id = "S1_CRASH_RESUME_DEMO"

mock_llm.reset()

try:

    crash_result = run_agent(

        user_message=s1_message,

        scenario_id="S1",

        max_iterations=6,

        token_limit=6000,

        time_limit_seconds=10,

        run_id=crash_demo_run_id,

        crash_after_tool_executions=2

    )

except RuntimeError as error:

    print("Simulated crash:", error)

Simulated crash: SIMULATED_CRASH_AFTER_CHECKPOINT


In [69]:
saved_state = load_checkpoint(
    crash_demo_run_id
)

print("Scenario:", saved_state["scenario_id"])

print("Tools executed:", saved_state["tools_executed"])

print("Number of completed steps:", len(saved_state["steps"]))

print("Model calls:", saved_state["llm_calls"])

print("Pending tool calls:", saved_state["pending_tool_calls"])

Scenario: S1
Tools executed: ['get_order', 'search_policy']
Number of completed steps: 2
Model calls: 2
Pending tool calls: [{'id': 'call_2', 'name': 'search_policy', 'arguments': {'query': 'refund policy late delivery'}}]


In [70]:
resumed_result = resume_agent(

    run_id=crash_demo_run_id,

    max_iterations=6,

    token_limit=6000,

    time_limit_seconds=10

)

print("Status:", resumed_result["status"])

print("Final answer:", resumed_result["final_answer"])

print("Model calls:", resumed_result["llm_calls"])

print("Tokens used:", resumed_result["tokens_used"])

Resuming run: S1_CRASH_RESUME_DEMO
Previously executed tools: ['get_order', 'search_policy']
Status: REFUND
Final answer: REFUND: INR 128.25 issued.
Model calls: 5
Tokens used: 1597


In [71]:
final_state = load_checkpoint(
    crash_demo_run_id
)

tools_across_both_runs = final_state["tools_executed"]

print(tools_across_both_runs)

['get_order', 'search_policy', 'calculate_refund', 'issue_refund']


In [72]:
expected_tool_sequence = [

    "get_order",

    "search_policy",

    "calculate_refund",

    "issue_refund"

]

assert tools_across_both_runs == expected_tool_sequence

assert len(tools_across_both_runs) == 4

assert resumed_result["status"] == "REFUND"

In [73]:
issued_amounts = [

    step["observation"]["refund_inr"]

    for step in resumed_result["steps"]

    if step["tool_name"] == "issue_refund"

]

assert len(issued_amounts) == 1

assert abs(
    issued_amounts[0] - 128.25
) <= 0.01

print("Crash-and-resume test passed.")

Crash-and-resume test passed.


In [74]:
readme_rows = []

for (_, row), result in zip(
    scenarios_df.iterrows(),
    scenario_results
):

    readme_rows.append({

        "scenario_id": str(row["scenario_id"]),

        "status": result["status"],

        "refund_amount": extract_refund_amount(result),

        "model_calls": result["llm_calls"],

        "tokens_used": result["tokens_used"]

    })


readme_results_df = pd.DataFrame(readme_rows)

display(readme_results_df)

,scenario_id,status,refund_amount,model_calls,tokens_used
0,S1,REFUND,128.25,5,1597
1,S2,REFUND,135.00,5,1597
2,S3,ESCALATE,1999.00,5,1584
3,S4,DENIED,0.00,4,1165
4,S5,DENIED,0.00,3,757
5,S6,NOT_FOUND,NaN,2,352
6,S7,FALLBACK_MAX_ITERATIONS,NaN,6,3840


In [75]:
def format_inr(value):

    if value is None:
        return "—"

    return f"₹{value:.2f}"


markdown_lines = [

    "| Scenario | Status | Refund Amount | Model Calls | Tokens Used |",

    "|---|---|---:|---:|---:|"

]

for _, row in readme_results_df.iterrows():

    refund = format_inr(
        row["refund_amount"]
    )

    markdown_lines.append(

        f"| {row['scenario_id']} "
        f"| {row['status']} "
        f"| {refund} "
        f"| {row['model_calls']} "
        f"| {row['tokens_used']} |"

    )


print("\n".join(markdown_lines))

| Scenario | Status | Refund Amount | Model Calls | Tokens Used |
|---|---|---:|---:|---:|
| S1 | REFUND | ₹128.25 | 5 | 1597 |
| S2 | REFUND | ₹135.00 | 5 | 1597 |
| S3 | ESCALATE | ₹1999.00 | 5 | 1584 |
| S4 | DENIED | ₹0.00 | 4 | 1165 |
| S5 | DENIED | ₹0.00 | 3 | 757 |
| S6 | NOT_FOUND | ₹nan | 2 | 352 |
| S7 | FALLBACK_MAX_ITERATIONS | ₹nan | 6 | 3840 |


In [76]:
print(
    "Token demonstration:",
    token_demo["status"]
)

print(
    "Timeout demonstration:",
    timeout_demo["status"]
)

Token demonstration: FALLBACK_TOKEN_LIMIT
Timeout demonstration: FALLBACK_TIMEOUT


In [77]:
print("Tools executed across crash and resume:")

for tool in tools_across_both_runs:

    print("-", tool)

print("Final status:", resumed_result["status"])

print("Final answer:", resumed_result["final_answer"])

Tools executed across crash and resume:
- get_order
- search_policy
- calculate_refund
- issue_refund
Final status: REFUND
Final answer: REFUND: INR 128.25 issued.


In [78]:
print(AGENT_RUNS_PATH.exists())

agent_logs = pd.read_json(
    AGENT_RUNS_PATH,
    lines=True
)

print("Number of logged runs:", len(agent_logs))

display(agent_logs)

True
Number of logged runs: 7


,scenario_id,status,final_answer,llm_calls,tokens_used,steps
0,S1,REFUND,REFUND: INR 128.25 issued.,5,1597,"[{'tool_name': 'get_order', 'arguments': {'ord..."
1,S2,REFUND,REFUND: INR 135.0 issued.,5,1597,"[{'tool_name': 'get_order', 'arguments': {'ord..."
2,S3,ESCALATE,ESCALATE: refund above INR 1000 sent to a huma...,5,1584,"[{'tool_name': 'get_order', 'arguments': {'ord..."
3,S4,DENIED,DENIED: delivery was within 45 minutes.,4,1165,"[{'tool_name': 'get_order', 'arguments': {'ord..."
4,S5,DENIED,DENIED: only completed orders can be refunded.,3,757,"[{'tool_name': 'get_order', 'arguments': {'ord..."
5,S6,NOT_FOUND,NOT_FOUND: no such order.,2,352,"[{'tool_name': 'get_order', 'arguments': {'ord..."
6,S7,FALLBACK_MAX_ITERATIONS,FALLBACK_MAX_ITERATIONS: The agent reached its...,6,3840,"[{'tool_name': 'search_policy', 'arguments': {..."


In [79]:
if REFUND_LEDGER_PATH.exists():

    ledger_df = pd.read_json(
        REFUND_LEDGER_PATH,
        lines=True
    )

    display(ledger_df)

else:

    print("Refund ledger does not exist.")

,timestamp,order_id,refund_inr,tool_name
0,2026-11-08 16:12:31.317301+00:00,ORD-100048,128.25,issue_refund
1,2026-11-08 16:12:31.354419+00:00,ORD-100067,135.00,issue_refund
2,2026-11-08 16:12:31.370717+00:00,ORD-100145,1999.00,escalate_to_human
3,2026-11-08 16:17:48.398392+00:00,ORD-100048,128.25,issue_refund
